In [131]:
"""
One-time script to preprocess raw UCSC XENA clinical data and save as a new csv
"""

'\nOne-time script to preprocess raw UCSC XENA clinical data and save as a new csv\n'

In [132]:
# Dataset is GDC TCGA BRCA via UCSC Xena
# Phenotype
# https://xenabrowser.net/datapages/?dataset=TCGA-BRCA.clinical.tsv&host=https%3A%2F%2Fgdc.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443

In [133]:
import os
import pandas as pd
import seaborn as sns

In [134]:
df = pd.read_csv("../data/raw/TCGA-BRCA.clinical.tsv", 
                  delimiter='\t', index_col=0)
df.head()

,id,disease_type,case_id,submitter_id,primary_site,alcohol_history.exposures,race.demographic,gender.demographic,ethnicity.demographic,vital_status.demographic,...,days_to_collection.samples,initial_weight.samples,preservation_method.samples,pathology_report_uuid.samples,oct_embedded.samples,specimen_type.samples,days_to_sample_procurement.samples,is_ffpe.samples,tissue_type.samples,annotations.samples
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-BH-A0W3-01A,3c612e12-6de8-44fa-a095-805c45474821,Ductal and Lobular Neoplasms,3c612e12-6de8-44fa-a095-805c45474821,TCGA-BH-A0W3,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,85.0,120.0,OCT,801A4E2F-E26E-424F-BF42-CD0D9CD62BCE,True,Solid Tissue,NaN,False,Tumor,NaN
TCGA-AR-A24V-01A,3cb06c7a-f2a8-448b-91a8-dd201bbf2ddd,Ductal and Lobular Neoplasms,3cb06c7a-f2a8-448b-91a8-dd201bbf2ddd,TCGA-AR-A24V,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,1720.0,400.0,OCT,468CD293-C9F7-43C6-A40A-18FCDD22F6AA,True,Solid Tissue,NaN,False,Tumor,NaN
TCGA-E9-A1NE-01A,3d676bba-154b-4d22-ab59-d4d4da051b94,Ductal and Lobular Neoplasms,3d676bba-154b-4d22-ab59-d4d4da051b94,TCGA-E9-A1NE,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,31.0,280.0,OCT,CF6E29A2-FAE6-45BB-B625-33877887A89E,True,Solid Tissue,NaN,False,Tumor,NaN
TCGA-E9-A1NE-11A,3d676bba-154b-4d22-ab59-d4d4da051b94,Ductal and Lobular Neoplasms,3d676bba-154b-4d22-ab59-d4d4da051b94,TCGA-E9-A1NE,Breast,Not Reported,white,female,not hispanic or latino,Alive,...,31.0,830.0,OCT,NaN,True,Solid Tissue,NaN,False,Normal,NaN
TCGA-AC-A8OQ-01A,dfaabd03-2d40-4422-b210-caf112ff4229,Ductal and Lobular Neoplasms,dfaabd03-2d40-4422-b210-caf112ff4229,TCGA-AC-A8OQ,Breast,Not Reported,black or african american,female,not hispanic or latino,Alive,...,742.0,100.0,Unknown,FFA6F9F3-71C1-4AF9-B9F7-0466550EBC90,False,Solid Tissue,NaN,False,Tumor,NaN


In [135]:
df.shape

(1255, 84)

In [136]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1255 entries, TCGA-BH-A0W3-01A to TCGA-A7-A0D9-11A
Data columns (total 84 columns):
 #   Column                                                     Non-Null Count  Dtype  
---  ------                                                     --------------  -----  
 0   id                                                         1255 non-null   object 
 1   disease_type                                               1255 non-null   object 
 2   case_id                                                    1255 non-null   object 
 3   submitter_id                                               1255 non-null   object 
 4   primary_site                                               1255 non-null   object 
 5   alcohol_history.exposures                                  1254 non-null   object 
 6   race.demographic                                           1254 non-null   object 
 7   gender.demographic                                         1254 non-null  

In [137]:
# STEP 1 - Only use primary tumor samples
# primary tumor is denoted as TCGA-xx-xxxx-01

rows_to_drop = [row for row in df.index if row[13:15] != '01']
df.drop(index=rows_to_drop, inplace=True)
df.head(15)
print(f"Dropped {len(rows_to_drop)} rows that were not primary tumors. {df.shape[0]} rows remaining.")

Dropped 144 rows that were not primary tumors. 1111 rows remaining.


In [138]:
# STEP 2 - drop cols that are >50% null

n_notnull = {}
for col in df.columns:
    n_notnull[col] = int(df[col].notnull().sum())

cols_to_drop = [col for col in n_notnull.keys()
                if n_notnull[col] < df.shape[0]*0.5]

print(cols_to_drop)

df.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped {len(cols_to_drop)} columns that had >50% nulls. {df.shape[1]} columns remaining.")

['year_of_death.demographic', 'days_to_death.demographic', 'entity_submitter_id.annotations', 'notes.annotations', 'submitter_id.annotations', 'classification.annotations', 'entity_id.annotations', 'created_datetime.annotations', 'annotation_id.annotations', 'entity_type.annotations', 'updated_datetime.annotations', 'case_id.annotations', 'state.annotations', 'category.annotations', 'status.annotations', 'case_submitter_id.annotations', 'days_to_sample_procurement.samples', 'annotations.samples']
Dropped 18 columns that had >50% nulls. 66 columns remaining.


In [139]:
# STEP 3 - drop cols that are >50% "Not Reported"

n_notreported = {}
for col in df.columns:
    n_notreported[col] = len([x for x in df[col] 
                              if x == "Not Reported" or x == "not reported"])

cols_to_drop = [col for col in n_notreported.keys()
                if n_notreported[col] > df.shape[0]*0.5]

print(cols_to_drop)

df.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped {len(cols_to_drop)} columns that had >50% Not Reported. {df.shape[1]} columns remaining.")

['alcohol_history.exposures', 'last_known_disease_status.diagnoses', 'classification_of_tumor.diagnoses', 'tumor_grade.diagnoses', 'progression_or_recurrence.diagnoses', 'composition.samples']
Dropped 6 columns that had >50% Not Reported. 60 columns remaining.


In [140]:
# STEP 4 - drop columns with only a single value

monotonic_cols = []
for col in df.columns:
    n_unique = len(df[col].unique())
    if n_unique == 1:
        monotonic_cols.append(col)

print(monotonic_cols)

df.drop(columns=monotonic_cols, inplace=True)
print(f"Dropped {len(monotonic_cols)} columns that had only one unique value. {df.shape[1]} columns remaining.")

['primary_site', 'primary_site.project', 'project_id.project', 'disease_type.project', 'name.project', 'name.program.project', 'project.tissue_source_site', 'bcr_id.tissue_source_site', 'sample_type_id.samples', 'tumor_descriptor.samples', 'sample_type.samples', 'specimen_type.samples', 'tissue_type.samples']
Dropped 13 columns that had only one unique value. 47 columns remaining.


In [141]:
# Examine distribution of variables

if not os.path.exists("../data/processed/clinical_distributions/"):
    os.mkdir("../data/processed/clinical_distributions/")

for col in df.columns:
    axes = sns.histplot(df, x=col)
    fig = axes.get_figure()
    fig.savefig(f"../data/processed/clinical_distributions/{col}.png")
    fig.clear()

<Figure size 640x480 with 0 Axes>

In [142]:
df.describe()

,age_at_index.demographic,days_to_birth.demographic,year_of_birth.demographic,days_to_diagnosis.diagnoses,days_to_last_follow_up.diagnoses,age_at_diagnosis.diagnoses,year_of_diagnosis.diagnoses,age_at_earliest_diagnosis.diagnoses.xena_derived,age_at_earliest_diagnosis_in_years.diagnoses.xena_derived,days_to_collection.samples,initial_weight.samples
count,1110.000000,1095.000000,1107.000000,1110.0,1006.000000,1095.000000,1108.000000,1095.000000,1095.000000,1109.000000,1098.000000
mean,58.492793,-21597.356164,1949.397471,0.0,1176.743539,21597.356164,2007.894404,21597.356164,59.170839,1058.921551,298.873406
std,13.176592,4799.676392,13.575691,0.0,1173.617563,4799.676392,4.288453,4799.676392,13.149798,1437.866373,262.640327
min,26.000000,-32872.000000,1902.000000,0.0,-7.000000,9706.000000,1988.000000,9706.000000,26.591781,16.000000,5.000000
25%,49.000000,-24862.500000,1940.000000,0.0,430.250000,18038.000000,2007.000000,18038.000000,49.419178,149.000000,130.000000
50%,58.000000,-21562.000000,1950.000000,0.0,760.000000,21562.000000,2009.000000,21562.000000,59.073973,447.000000,210.000000
75%,67.000000,-18038.000000,1960.000000,0.0,1558.250000,24862.500000,2010.000000,24862.500000,68.116438,1357.000000,370.000000
max,90.000000,-9706.000000,1984.000000,0.0,8605.000000,32872.000000,2013.000000,32872.000000,90.060274,7858.000000,2190.000000
